In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Gold intermediate source
price_quality = spark.table(
    get_gold_table("intermediate", "int_item_price_quality")
)

# Silver item dimension
dim_item = spark.table(
    get_table("intermediate", "dim_item")
)

print(f"Price quality records: {price_quality.count():,}")
print(f"Item dimension records: {dim_item.count():,}")

display(dim_item)

In [0]:
# Select item attributes
item_attributes = dim_item.select(
    "item_id",
    "item_name",
    "category",
    "is_set_root",
    "set_part_ids",
    "ducats",
    "required_mastery_rank"
)

# Enrich price observations
mart_item_price_history = (
    price_quality
    .join(
        item_attributes,
        on="item_id",
        how="left"
    )
)

print(f"Source records: {price_quality.count():,}")
print(f"Mart records: {mart_item_price_history.count():,}")

display(mart_item_price_history.limit(10))

In [0]:
# Validate mart enrichment
validation = mart_item_price_history.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.sum(
        F.when(F.col("item_name").isNull(), 1).otherwise(0)
    ).alias("missing_item_names"),
    F.sum(
        F.when(F.col("category").isNull(), 1).otherwise(0)
    ).alias("missing_categories"),
    F.sum(
        F.when(F.col("is_set_root").isNull(), 1).otherwise(0)
    ).alias("missing_set_flags")
)

display(validation)

In [0]:
# Build item price history mart
mart_item_price_history = (
    mart_item_price_history
    .select(
        # Observation identifiers
        "execution_id",
        "order_id",
        "item_id",

        # Item attributes
        "item_slug",
        "item_name",
        "product_slug",
        "category",
        "is_set_root",

        # Observation timestamps
        "observed_at",
        "order_created_at",
        "order_updated_at",

        # Offer attributes
        "order_type",
        "platinum",
        "quantity",
        "per_trade",

        # Player attributes
        "player_id",
        "player_name",
        "player_status",
        "platform",
        "crossplay",
        "locale",
        "player_reputation",

        # Price quality
        "price_quality_status",
        "is_price_outlier",
        "total_orders",
        "q1",
        "q3",
        "iqr",
        "lower_bound",
        "upper_bound",
        "quality_status",

        # Item metadata
        "set_part_ids",
        "ducats",
        "required_mastery_rank"
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp()
    )
)

display(mart_item_price_history.limit(10))

In [0]:
# Gold target
MART_TARGET = get_gold_table(
    "marts",
    "mart_item_price_history"
)

# Materialize Delta table
(
    mart_item_price_history.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MART_TARGET)
)

print(f"Table successfully materialized: {MART_TARGET}")

In [0]:
# Validate materialized mart
mart_result = spark.table(MART_TARGET)

print(f"Total records: {mart_result.count():,}")

display(
    mart_result
    .groupBy("price_quality_status")
    .count()
    .orderBy("price_quality_status")
)